# 03 — Data Cleaning

**AttriSense · Employee Attrition Prediction & Analytics System**

Cleaning transforms raw CSV data into a validated dataset stored at `data/processed/`. Downstream notebooks (EDA, feature engineering, modeling) read from that file — they never touch the raw CSV directly.

Cleaning logic lives in `src/attrisense/data/cleaning.py` so the pipeline is reusable outside notebooks.

In [ ]:
from attrisense.config import load_config
from attrisense.data import (
    load_raw_data,
    clean_raw_data,
    save_processed_data,
    load_processed_data,
    drop_uninformative_columns,
)

config = load_config()

## 1. Load Raw Data

Same entry point as the dataset understanding notebook. Keeping a single loader function avoids path drift between notebooks.

In [ ]:
raw_df = load_raw_data()
print(f"Raw shape: {raw_df.shape}")
raw_df.head(3)

## 2. Drop Uninformative Columns

Three columns are constant across every row:

| Column | Value | Reason to drop |
|--------|-------|----------------|
| `EmployeeCount` | 1 | No variance |
| `Over18` | Y | No variance |
| `StandardHours` | 80 | No variance |

These are listed in `configs/config.yaml` under `drop_columns`. Including them in a one-hot encoding step would create zero-variance features that add noise without signal.

In [ ]:
after_drop = drop_uninformative_columns(raw_df, config=config)
dropped = set(raw_df.columns) - set(after_drop.columns)

print(f"Dropped {len(dropped)} columns: {sorted(dropped)}")
print(f"Shape after drop: {after_drop.shape}")

## 3. Validate Data Quality

Even though the dataset understanding notebook showed no missing values, we validate programmatically. If someone swaps in a different CSV later, the pipeline should fail loudly rather than silently produce bad models.

Checks performed:
- No null values in any column
- Target column contains only `"Yes"` and `"No"`

In [ ]:
# clean_raw_data() runs drop + validation in one call
cleaned_df = clean_raw_data(raw_df, config=config)

print(f"Cleaned shape: {cleaned_df.shape}")
print(f"Target values: {sorted(cleaned_df[config.target_column].unique())}")
print(f"Null count    : {cleaned_df.isna().sum().sum()}")

## 4. What We Deliberately Keep

| Column | Decision | Rationale |
|--------|----------|-----------|
| `EmployeeNumber` | Keep in dataset, exclude from features later | Unique ID for traceability |
| `Attrition` | Keep as target | Binary label for classification |
| All other columns | Keep | Candidate features for EDA and modeling |

No rows are removed. The dataset has no duplicates and no obvious data entry errors that warrant row-level deletion at this stage.

## 5. Save Processed Dataset

Cleaned data is written to Parquet format in `data/processed/`. Parquet preserves dtypes, compresses well, and loads faster than CSV for repeated notebook runs.

In [ ]:
output_path = save_processed_data(cleaned_df, config=config)
print(f"Saved to: {output_path}")

# Round-trip check: load back and confirm shape matches
reloaded = load_processed_data(config=config)
assert reloaded.shape == cleaned_df.shape, "Round-trip shape mismatch"
print(f"Round-trip verified: {reloaded.shape}")

## 6. Cleaning Summary

| Step | Action | Result |
|------|--------|--------|
| Load | Read raw CSV | 1,470 × 35 |
| Drop constants | Remove 3 zero-variance columns | 1,470 × 32 |
| Validate | Check nulls and target labels | Pass |
| Save | Write Parquet to `data/processed/` | Reusable artifact |

**Next step:** `04_eda.ipynb` — exploratory analysis on the cleaned dataset.